# 🎯 TESTING

**Pipeline:**
1. Tracking de 10 modèles (4 baseline + 4 tuned + 2 stacking)
2. Lecture des résultats avec Pandas
3. Sélection du meilleur modèle (ROC-AUC)
4. Chargement et utilisation du modèle
5. Enregistrement dans Model Registry (local)

In [ ]:
import pandas as pd
pd.options.display.max_columns=None
import numpy as np

In [ ]:
df = pd.read_csv("../data/raws/fraud_600k.csv")
df.drop(['Unnamed: 0'],axis=1,inplace =True)
df.head()

In [ ]:
import deepchecks
deepchecks.__version__

full_suite() est une collection pré-configurée de tests créée par Deepchecks
C'est comme une "checklist automatique" de qualité des données

In [ ]:
from deepchecks.tabular.suites import full_suite
suites = full_suite()
suites

Deepchecks a besoin de savoir quel type de données il analyse
Les colonnes catégorielles (type object en pandas) doivent être traitées différemment des colonnes numériques
select_dtypes(include=['object']) = "donne-moi toutes les colonnes de type texte/string"

In [ ]:
cat_features = df.select_dtypes(include=['object']).columns.tolist()
cat_features

Pandas ne sait pas quelle colonne est la cible (ce qu'on veut prédire)
Pandas ne fait pas la différence entre features catégorielles et numériques de manière intelligente
Le Dataset de Deepchecks ajoute cette intelligence aux données

In [ ]:
from deepchecks.tabular import Dataset
dataframeTest=Dataset(df=df,label='is_fraud',cat_features=cat_features)
dataframeTest

Lance l'exécution de TOUS les tests de la suite complète sur ton dataset. Génère un rapport interactif

In [ ]:
suites.run(dataframeTest)

Un dataset plus clean qui contient seulement les features qui ont du sens pour détecter la fraude (category, gender..)

In [ ]:
dfcopy=df.copy()
dfcopy.drop(['trans_date_trans_time','first', 'last', 'dob','trans_num','cc_num','unix_time','merchant', 'street', 'city','lat','long','state'] , axis=1, inplace=True)
#get categorical features
cat_features= dfcopy.select_dtypes(include=['object']).columns.tolist()
dataframeTest=Dataset(df=df,label='is_fraud',cat_features=cat_features)
dataframeTest

In [ ]:
suites.run(dataframeTest)

## Individual Test

Au lieu d'utiliser la suite complète (full_suite), on importe une suite SPÉCIFIQUE qui teste uniquement l'intégrité des données (valeurs manquantes, duplicates, types de données, etc.). C'est plus ciblé et plus rapide.

In [ ]:
from deepchecks.suites import data_integrity

In [ ]:
integrity_test=data_integrity()
integrity_test

In [ ]:
result_integrity_test=integrity_test.run(dataframeTest)
result_integrity_test

## Saving result as html file

In [ ]:
# Sauvegarder le HTML
result_integrity_test.save_as_html('../artifacts/deepchecks_test/result_integrity_test.html')

# Lire le fichier
with open('../artifacts/deepchecks_test/result_integrity_test.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

# Scripts complets pour les widgets
scripts = '''
<script src="https://cdnjs.cloudflare.com/ajax/libs/require.js/2.3.6/require.min.js"></script>
<script src="https://unpkg.com/@jupyter-widgets/html-manager@1.0.9/dist/embed-amd.js"></script>
'''

# Insérer après <head>
html_content = html_content.replace('<head>', '<head>' + scripts)

# Réécrire
with open('../artifacts/deepchecks_test/result_integrity_test.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("✓ HTML généré")


## Data Distribution tests

In [ ]:
from sklearn.model_selection import train_test_split
df_train,df_test=train_test_split(dfcopy,test_size=0.2, random_state=42)

In [ ]:
print(f"df_train:{df_train.shape}")
print(f"df_test:{df_test.shape}")

On crée deux datasets Deepchecks séparés (train et test), puis on lance la suite complète de tests pour comparer les deux. L'objectif est de détecter le data drift (les distributions train vs test sont-elles différentes ?) et vérifier que le split train/test est cohérent.

In [ ]:
#transformation
ds_train=Dataset(df=df_train,label='is_fraud',cat_features=cat_features)
ds_test=Dataset(df=df_test,label='is_fraud',cat_features=cat_features)
data_distribution_test=suites.run(train_dataset=ds_train,test_dataset=ds_test)
data_distribution_test

In [ ]:
# Sauvegarder le HTML
data_distribution_test.save_as_html('../artifacts/deepchecks_test/data_distribution_test.html')

# Lire le fichier
with open('../artifacts/deepchecks_test/data_distribution_test.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

# Scripts complets pour les widgets
scripts = '''
<script src="https://cdnjs.cloudflare.com/ajax/libs/require.js/2.3.6/require.min.js"></script>
<script src="https://unpkg.com/@jupyter-widgets/html-manager@1.0.9/dist/embed-amd.js"></script>
'''

# Insérer après <head>
html_content = html_content.replace('<head>', '<head>' + scripts)

# Réécrire
with open('../artifacts/deepchecks_test/data_distribution_test.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("✓ HTML généré")

## Model analysis and validation tests

On crée une nouvelle copie du dataset spécifiquement pour entraîner et tester un modèle. On supprime encore plus de colonnes que dans les tests précédents, en gardant SEULEMENT les features qui seront utilisées par le modèle ML.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import  train_test_split

dfModel=df.copy()
dfModel.drop(['trans_date_trans_time','first', 'last', 'dob','trans_num','cc_num','unix_time','merchant', 'street', 'city','lat','long','state','job'] , axis=1, inplace=True)
dfModelCopy=dfModel.copy()
#get categorical features
cat_features= dfModel.select_dtypes(include=['object']).columns.tolist()
num_features= dfModel.select_dtypes(include=['integer']).columns.tolist()
num_features.remove("is_fraud")

encoder les valeurs categorielle : 'F': 0, 'M': 1

In [ ]:
#Only for gender
encode_dict = {  # Encoding dictionary
    'F': 0, 'M': 1}
dfModel['gender'] = dfModel['gender'].map(encode_dict)
dummy_cols = ['category']
#For other categorical columns
dfModel = pd.get_dummies(dfModel, columns=dummy_cols)
dfModel.head()

In [ ]:
X = dfModel.drop('is_fraud', axis=1)  # Select features
y = dfModel['is_fraud']  # Target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1)  # Split data 80/20

In [ ]:
# Normalize train & test features
scaler = MinMaxScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

## Smote

In [ ]:
from imblearn.over_sampling import SMOTE

method= SMOTE()
x_resampled, y_resampled = method.fit_resample(X_train, y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression

model=LogisticRegression(solver='lbfgs' ,max_iter=10000)
model.fit(x_resampled,y_resampled)

In [ ]:
df_train,df_test=train_test_split(dfModel,test_size=0.2)
df_train_model=Dataset(df=df_train,label='is_fraud')
df_test_model=Dataset(df=df_test,label='is_fraud')

In [ ]:
result_model_test=suites.run(train_dataset=df_train_model,test_dataset=df_test_model,model=model)

In [ ]:
result_model_test

In [ ]:
# Sauvegarder le HTML
result_model_test.save_as_html('../artifacts/deepchecks_test/result_model_test.html')

# Lire le fichier
with open('../artifacts/deepchecks_test/result_model_test.html', 'r', encoding='utf-8') as f:
    html_content = f.read()

# Scripts complets pour les widgets
scripts = '''
<script src="https://cdnjs.cloudflare.com/ajax/libs/require.js/2.3.6/require.min.js"></script>
<script src="https://unpkg.com/@jupyter-widgets/html-manager@1.0.9/dist/embed-amd.js"></script>
'''

# Insérer après <head>
html_content = html_content.replace('<head>', '<head>' + scripts)

# Réécrire
with open('../artifacts/deepchecks_test/result_model_test.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print("✓ HTML généré")